# Pipeline de dados do Frigus

## Objetivo
Ler os dados da aplicação no Supabase, armazenar uma cópia
na camada Raw e produzir tabelas tratadas na camada Cleaned

## Fluxo
CSV exportado do Supabase --> workspace.frigus_raw --> workspace.frigus_cleaned

## Etapas
1. Leitura das tabelas da aplicação
2. Gravação em Delta na camada Raw
3. Remoção de registros integralmente duplicados
4. Identificação de valores nulos
5. Remoção de espaços nas extremidades dos campos textuais
6. Inclusão da data de tratamento
7. Gravação das tabelas tratadas
8. Validação dos resultados

## Ingestão de Dados (Source -> Raw)

In [0]:
from pyspark.sql import functions as F

PASTA_ENTRADA = ("/Volumes/workspace/frigus_arquivos/entrada")

TABELAS = [
    "transactions",
    "plans",
    "users",
    "subscriptions",
    "discard",
    "stock_movements",
    "products",
    "stock_products",
    "stocks",
    "groups"
]

for tabela in TABELAS:
    caminho = f"{PASTA_ENTRADA}/{tabela}.csv"

    print(f"Lendo arquivo: {tabela}.csv")

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("multiLine", "true")
        .option("escape", '"')
        .option("mode", "FAILFAST")
        .csv(caminho))

    if tabela == "transactions":
        df = (
            df
            .withColumn(
                "created_at",
                F.to_timestamp("created_at"))
            .withColumn(
                "amount",
                F.col("amount").cast("decimal(10,2)"))
            .withColumn(
                "attempts",
                F.col("attempts").cast("int"))
        )

    df = df.withColumn(
        "_ingested_at",
        F.current_timestamp())
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            f"workspace.frigus_raw.{tabela}"
        )
    )

    print(f"Ingestao concluida: {tabela}")

Lendo arquivo: transactions.csv
Ingestao concluida: transactions
Lendo arquivo: plans.csv
Ingestao concluida: plans
Lendo arquivo: users.csv
Ingestao concluida: users
Lendo arquivo: subscriptions.csv
Ingestao concluida: subscriptions
Lendo arquivo: discard.csv
Ingestao concluida: discard
Lendo arquivo: stock_movements.csv
Ingestao concluida: stock_movements
Lendo arquivo: products.csv
Ingestao concluida: products
Lendo arquivo: stock_products.csv
Ingestao concluida: stock_products
Lendo arquivo: stocks.csv
Ingestao concluida: stocks
Lendo arquivo: groups.csv
Ingestao concluida: groups


In [0]:
from pyspark.sql import functions as F
from datetime import datetime
import time

RAW_SCHEMA = "workspace.frigus_raw"
CLEANED_SCHEMA = "workspace.frigus_cleaned"

In [0]:
def processar_tabela(table_name, verbose=True):
    """
    Processa uma tabela raw para cleaned com limpeza e transformações.
    
    Args:
        table_name: Nome da tabela (sem schema)
        verbose: Se True, imprime logs detalhados
    
    Returns:
        dict com estatísticas do processamento
    """
    start_time = time.time()
    
    try:
        raw_table = f"{RAW_SCHEMA}.{table_name}"
        cleaned_table = f"{CLEANED_SCHEMA}.{table_name}"
        
        if verbose:
            print(f"\n[PROCESSANDO] {table_name}")
            print("-" * 60)
        
        # Leitura da tabela raw
        df_raw = spark.table(raw_table)
        total_records = df_raw.count()
        
        if verbose:
            print(f"  [LEITURA] {total_records:,} registros")
        
        # Remoção de duplicatas
        df_cleaned = df_raw.dropDuplicates()
        distinct_records = df_cleaned.count()
        duplicates_removed = total_records - distinct_records
        
        if verbose:
            print(f"  [DUPLICATAS] {duplicates_removed:,} removidas")
        
        # Análise de nulos OTIMIZADA, uma única agregação para todas as colunas
        schema_fields = df_cleaned.schema.fields  # Cache schema antes do loop
        columns_list = [f.name for f in schema_fields]  # Cache lista de colunas
        
        # Criar agregações para contar nulos em todas as colunas de uma vez
        null_aggs = [
            F.sum(
                F.when(F.col(c).isNull(), 1)
                .otherwise(0)).alias(f"{c}_nulls") 
                for c in columns_list
        ]
        null_counts = df_cleaned.agg(*null_aggs).collect()[0].asDict()
        
        # Processar resultados
        null_columns = []
        for col_name in columns_list:
            null_count = null_counts[f"{col_name}_nulls"] or 0
            if null_count > 0:
                null_pct = (null_count / distinct_records * 100) if distinct_records > 0 else 0
                null_columns.append(f"{col_name} ({null_pct:.1f}%)")
        
        if verbose and null_columns:
            print(f"  [NULOS] {len(null_columns)} colunas com nulos")
        
        # Limpeza de strings, usar withColumns ao invés de múltiplos withColumn
        string_cols = [ f.name for f in schema_fields if f.dataType.simpleString().split("(")[0] in ("string", "varchar", "char")]
        if string_cols:
            trim_cols = {col: F.trim(F.col(col)) for col in string_cols}
            df_cleaned = df_cleaned.withColumns(trim_cols)
        
        if verbose:
            print(f"  [STRINGS] {len(string_cols)} colunas trimadas")
        
        # Adicionar colunas de auditoria
        df_cleaned = df_cleaned.withColumn("_cleaned_at", F.current_timestamp())
        
        # Gravação na cleaned
        df_cleaned.write \
            .format("delta") \
            .mode("overwrite") \
            .option("mergeSchema", "true") \
            .saveAsTable(cleaned_table)
        
        # Validação
        cleaned_count = spark.table(cleaned_table).count()
        
        elapsed_time = time.time() - start_time
        
        if verbose:
            print(f"  [GRAVACAO] {cleaned_count:,} registros em {cleaned_table}")
            print(f"  [TEMPO] {elapsed_time:.2f}s")
            print(f"  [STATUS] SUCESSO")
        
        return {
            "table_name": table_name,
            "status": "success",
            "raw_records": total_records,
            "cleaned_records": cleaned_count,
            "duplicates_removed": duplicates_removed,
            "null_columns": len(null_columns),
            "elapsed_time": elapsed_time,
            "error": None
        }
        
    except Exception as e:
        elapsed_time = time.time() - start_time
        
        if verbose:
            print(f"  [ERRO] {str(e)}")
            print(f"  [TEMPO] {elapsed_time:.2f}s")
            print(f"  [STATUS] FALHA")
        
        return {
            "table_name": table_name,
            "status": "error",
            "raw_records": 0,
            "cleaned_records": 0,
            "duplicates_removed": 0,
            "null_columns": 0,
            "elapsed_time": elapsed_time,
            "error": str(e)
        }

In [0]:
# Tratar exatamente as tabelas ingeridas nesta execução.
table_names = list(TABELAS)

print(f"Tabelas selecionadas: {len(table_names)}")

for nome in table_names:
    print(nome)

Tabelas selecionadas: 10
transactions
plans
users
subscriptions
discard
stock_movements
products
stock_products
stocks
groups


In [0]:
# Processar todas as tabelas
results = []
for table_name in table_names:
    result = processar_tabela(table_name, verbose=True)
    results.append(result)

In [0]:
# Gerar relatório consolidado
print("\n" + "="*80)
print("RELATORIO FINAL - LIMPEZA RAW -> CLEANED")
print("="*80)

success_count = sum(1 for r in results if r["status"] == "success")
error_count = sum(1 for r in results if r["status"] == "error")
total_raw = sum(r["raw_records"] for r in results)
total_cleaned = sum(r["cleaned_records"] for r in results)
total_duplicates = sum(r["duplicates_removed"] for r in results)
total_time = sum(r["elapsed_time"] for r in results)

print(f"\nRESUMO GERAL:")
print(f"Tabelas processadas: {len(results)}")
print(f"Sucessos: {success_count}")
print(f"Erros: {error_count}")
print(f"Registros raw: {total_raw:,}")
print(f"Registros cleaned: {total_cleaned:,}")
print(f"Duplicatas removidas: {total_duplicates:,}")
print(f"Tempo total: {total_time:.2f}s")
print(f"Tempo medio por tabela: {total_time/len(results):.2f}s")

if error_count > 0:
    print(f"\nERROS:")
    for r in results:
        if r["status"] == "error":
            print(f"  - {r['table_name']}: {r['error']}")

print(f"\nDETALHES POR TABELA:")
print(f"{'Tabela':<30} {'raw':>12} {'cleaned':>12} {'Dup.Rem':>10} {'Tempo':>8} {'Status'}")
print("-" * 90)

for r in results:
    status_symbol = "OK" if r["status"] == "success" else "ERRO"
    print(f"{r['table_name']:<30} {r['raw_records']:>12,} {r['cleaned_records']:>12,} {r['duplicates_removed']:>10,} {r['elapsed_time']:>7.1f}s {status_symbol}")

print("="*80)
print(f"Termino: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

In [0]:
falhas = [
    resultado
    for resultado in results
    if resultado.get("status") != "success"
]

if falhas:
    detalhes = "; ".join(
        f"{resultado['table_name']}: "
        f"{resultado.get('error')}"
        for resultado in falhas
    )
    raise RuntimeError(
        f"Tratamento incompleto: {detalhes}"
    )

print("Validação deu certo, todas as tabelas foram tratadas")